# Быстрая модель: ориентация текста 0°/180°

Этот ноутбук самостоятельно создаёт `submission.csv` для всех 20 000 тестовых кропов.
Используется локальная PaddleOCR `PP-LCNet_x1_0_textline_ori`: каждый кроп проверяется
в исходном положении и после поворота на 180°. Итоговая вероятность — среднее двух
ответов в логарифмических шансах. Модель работает на CPU, как в ранее отправленном
варианте. После запуска выводится время загрузки, обработки и полное время.

Запускайте ноутбук из корня проекта, где есть `models/` и `test_data/` (или `test.zip`).
Другой корень можно указать через `AVITO_PROJECT_DIR`, другой путь к тесту — через
`AVITO_TEST_PATH`. Установка для Python 3.13: `pip install paddleocr==3.7.0
paddlepaddle==3.3.0 numpy==2.3.5 Pillow==12.1.0`. Веса уже лежат в `models/`;
внешние API и тестовые метки не используются.

In [ ]:
from __future__ import annotations

import csv
import io
import math
import os
import time
import zipfile
from pathlib import Path

import sys
# These notebooks use Paddle only; a broken optional Torch installation can block PaddleX.
sys.modules['torch'] = None


import numpy as np
from PIL import Image
from paddleocr import TextLineOrientationClassification

PROJECT_DIR = Path(os.environ.get("AVITO_PROJECT_DIR", Path.cwd())).expanduser().resolve()
MODEL_DIR = PROJECT_DIR / "models" / "PP-LCNet_x1_0_textline_ori_infer"
OUTPUT_CSV = PROJECT_DIR / "submission.csv"
DEVICE = "cpu"
BATCH_PAIRS = 16

if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f"Model weights not found: {MODEL_DIR}")
print(f"Project: {PROJECT_DIR}")
print(f"Classifier device: {DEVICE}")

In [ ]:
EXPECTED_ROWS = 20_000

def find_test_data(project_dir: Path) -> Path:
    explicit = os.environ.get("AVITO_TEST_PATH")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    for path in (project_dir / "test_data", project_dir / "test.zip"):
        if path.exists():
            return path
    raise FileNotFoundError("Set AVITO_TEST_PATH or put test_data/test.zip next to the notebook")

class TestData:
    """Read test images in exactly the order of sample_submission.csv."""

    def __init__(self, path: Path):
        self.archive = None
        self.image_dir = None
        if path.is_file():
            self.archive = zipfile.ZipFile(path)
            sample = self.archive.read("sample_submission.csv")
        elif path.is_dir():
            if (path / "test" / "images").is_dir():
                self.image_dir = path / "test" / "images"
                sample_path = path / "sample_submission.csv"
            elif (path / "images").is_dir():
                self.image_dir = path / "images"
                sample_path = path / "sample_submission.csv"
                if not sample_path.is_file():
                    sample_path = path.parent / "sample_submission.csv"
            else:
                raise FileNotFoundError(f"No test/images directory in {path}")
            sample = sample_path.read_bytes()
        else:
            raise FileNotFoundError(path)
        reader = csv.DictReader(io.StringIO(sample.decode("utf-8-sig")))
        if reader.fieldnames != ["image_id", "p_180"]:
            raise ValueError(f"Unexpected sample columns: {reader.fieldnames}")
        self.image_ids = [row["image_id"] for row in reader]
        if len(self.image_ids) != EXPECTED_ROWS or len(set(self.image_ids)) != EXPECTED_ROWS:
            raise ValueError("Expected 20,000 unique image IDs")

    def __enter__(self):
        return self

    def __exit__(self, *_):
        if self.archive is not None:
            self.archive.close()

    def read_image(self, image_id: str) -> bytes:
        name = f"{image_id}.png"
        if self.archive is not None:
            return self.archive.read(f"test/images/{name}")
        return (self.image_dir / name).read_bytes()

    def check_images(self) -> None:
        if self.archive is not None:
            available = set(self.archive.namelist())
            missing = [name for name in self.image_ids if f"test/images/{name}.png" not in available]
        else:
            missing = [name for name in self.image_ids if not (self.image_dir / f"{name}.png").is_file()]
        if missing:
            raise FileNotFoundError(f"Missing test image: {missing[0]}")

In [ ]:
PROBABILITY_LIMIT = 1e-6

def probability_180(result) -> float:
    label = result["label_names"][0]
    score = float(result["scores"][0])
    if label not in ("0_degree", "180_degree") or not math.isfinite(score):
        raise ValueError(f"Unexpected classifier output: {label}, {score}")
    if not 0 <= score <= 1:
        raise ValueError(f"Invalid classifier confidence: {score}")
    return score if label == "180_degree" else 1 - score

def logit(value: float) -> float:
    value = min(1 - PROBABILITY_LIMIT, max(PROBABILITY_LIMIT, value))
    return math.log(value / (1 - value))

def sigmoid(value: float) -> float:
    if value >= 0:
        return 1 / (1 + math.exp(-value))
    exp_value = math.exp(value)
    return exp_value / (1 + exp_value)

def baseline_probability(original: float, rotated: float) -> float:
    return sigmoid((logit(original) - logit(rotated)) / 2)

In [ ]:
total_start = time.perf_counter()
with TestData(find_test_data(PROJECT_DIR)) as test:
    test.check_images()

    start = time.perf_counter()
    model = TextLineOrientationClassification(
        model_name="PP-LCNet_x1_0_textline_ori",
        model_dir=str(MODEL_DIR),
        device=DEVICE,
    )
    model_load_seconds = time.perf_counter() - start

    image_seconds = 0.0
    inference_seconds = 0.0
    rows = []
    for start_index in range(0, len(test.image_ids), BATCH_PAIRS):
        ids = test.image_ids[start_index:start_index + BATCH_PAIRS]

        start = time.perf_counter()
        images = []
        for image_id in ids:
            with Image.open(io.BytesIO(test.read_image(image_id))) as source:
                image = source.convert("RGB")
            images.append(np.asarray(image))
            images.append(np.asarray(image.transpose(Image.Transpose.ROTATE_180)))
        image_seconds += time.perf_counter() - start

        start = time.perf_counter()
        predictions = model.predict(images, batch_size=2 * BATCH_PAIRS)
        inference_seconds += time.perf_counter() - start
        if len(predictions) != len(images):
            raise RuntimeError("Unexpected number of classifier predictions")

        for position, image_id in enumerate(ids):
            p0 = probability_180(predictions[2 * position])
            p180 = probability_180(predictions[2 * position + 1])
            rows.append((image_id, baseline_probability(p0, p180)))

        done = start_index + len(ids)
        if done % 1024 == 0 or done == len(test.image_ids):
            print(f"Processed {done}/{len(test.image_ids)}", flush=True)

    if [image_id for image_id, _ in rows] != test.image_ids:
        raise ValueError("Output IDs differ from sample_submission.csv")

start = time.perf_counter()
temporary = OUTPUT_CSV.with_suffix(".csv.tmp")
with temporary.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("image_id", "p_180"))
    writer.writerows((image_id, f"{p:.8f}") for image_id, p in rows)
os.replace(temporary, OUTPUT_CSV)
csv_seconds = time.perf_counter() - start
total_seconds = time.perf_counter() - total_start

print(f"Saved {len(rows)} predictions to {OUTPUT_CSV}")
print(f"Model load: {model_load_seconds:.2f} s")
print(f"Read and rotate: {image_seconds:.2f} s")
print(f"Classifier inference: {inference_seconds:.2f} s")
print(f"CSV write: {csv_seconds:.2f} s")
print(f"Total: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Throughput: {len(rows) / total_seconds:.1f} crops/s")